# CloudGuard-Water: Phase 4.1 — Dataset Provenance and Evaluation Audit

**Course**: Machine Learning for Cloud Services  
**Project**: CloudGuard-Water: Intelligent and Explainable Water Quality Classification  
**Target File**: `data/water_quality_dataset1.csv`  

---

### Objectives of this Audit:
1. **Canonical Dataset Forensics**: Independently compute and verify the exact byte size, SHA-256 checksum, column schema, missing values, duplicates, class distributions, and exact class-transition index.
2. **Reconcile Past Inconsistencies**: Investigate and document why past reports cited differing file sizes (2.83 MB vs. 4.17 MB) and inverted class block orderings.
3. **Empirical Rule vs. Statutory Standards Audit**: Verify the $TDS \le 211.0$, $\text{Turbidity} \le 4.45$, $6.5 \le pH \le 7.8$ rule, audit the 14 boundary edge cases, and rigorously distinguish empirical dataset thresholds from official WHO, BIS, and EPA drinking-water standards.
4. **Methodological Correction of Cross-Validation**: Re-run 5-fold Stratified Cross-Validation **strictly on the 80% training partition ($N=80,640$)**, reserving the 20% test partition ($N=20,160$) as a completely isolated final evaluation set.
5. **Class Weighting Re-Assessment**: Verify the impact of `class_weight='balanced'` on the training partition cross-validation folds.


In [1]:
import os
import sys
import hashlib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.exceptions import UndefinedMetricWarning

warnings.filterwarnings('ignore', category=UndefinedMetricWarning)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 130
sns.set_theme(style="whitegrid", palette="muted")


## 1. Forensic Verification of the Authoritative Dataset

We read the raw bytes of `data/water_quality_dataset1.csv` and compare it against its backup twin `data/water_quality_dataset1_original.csv`.


In [2]:
data_path = Path("../data/water_quality_dataset1.csv")
if not data_path.exists():
    data_path = Path("data/water_quality_dataset1.csv")

raw_bytes = data_path.read_bytes()
file_size = len(raw_bytes)
sha256 = hashlib.sha256(raw_bytes).hexdigest()

df = pd.read_csv(data_path)

print(f"File Path        : {data_path.resolve()}")
print(f"File Size        : {file_size:,} bytes ({file_size / (1024*1024):.4f} MB)")
print(f"SHA-256 Checksum : {sha256}")
print(f"Dataset Shape    : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns          : {list(df.columns)}")
print(f"Missing Values   : {df.isnull().sum().sum()}")
print(f"Duplicate Rows   : {df.duplicated().sum()}")

# Class counts and block layout audit
counts = df['Label'].value_counts()
print(f"\nClass Counts:")
print(f"  Safe   (0) : {counts['Safe']:,} ({counts['Safe']/len(df)*100:.2f}%)")
print(f"  Unsafe (1) : {counts['Unsafe']:,} ({counts['Unsafe']/len(df)*100:.2f}%)")

first_label = df['Label'].iloc[0]
last_label = df['Label'].iloc[-1]
print(f"\nRow 0 Label      : '{first_label}'")
print(f"Row 100799 Label : '{last_label}'")

# Detect transition point
s = df['Label']
transitions = (s != s.shift()).to_numpy().nonzero()[0]
print(f"Transitions indices: {list(transitions)}")
for t in transitions:
    prev = s.iloc[t-1] if t > 0 else 'START'
    print(f"  Row {t:>6}: Transition from '{prev}' -> '{s.iloc[t]}'")

print(f"\n-> VERIFIED: Rows 0 to 30,239 are 'Safe' ({counts['Safe']:,} rows).")
print(f"-> VERIFIED: Rows 30,240 to 100,799 are 'Unsafe' ({counts['Unsafe']:,} rows).")


File Path        : D:\E0224033\ML_for_Cl_Ser\CL\LB_Proj\data\water_quality_dataset1.csv
File Size        : 2,964,409 bytes (2.8271 MB)
SHA-256 Checksum : 90f9c7cb7404fb69b655ee27293c91b82ee3a346712e06035f8259393835a04a
Dataset Shape    : 100,800 rows × 5 columns
Columns          : ['pH', 'TDS_ppm', 'Turbidity_NTU', 'Temperature_C', 'Label']
Missing Values   : 0
Duplicate Rows   : 0

Class Counts:
  Safe   (0) : 30,240 (30.00%)
  Unsafe (1) : 70,560 (70.00%)

Row 0 Label      : 'Safe'
Row 100799 Label : 'Unsafe'
Transitions indices: [0, 30240]
  Row      0: Transition from 'START' -> 'Safe'
  Row  30240: Transition from 'Safe' -> 'Unsafe'

-> VERIFIED: Rows 0 to 30,239 are 'Safe' (30,240 rows).
-> VERIFIED: Rows 30,240 to 100,799 are 'Unsafe' (70,560 rows).


## 2. Descriptive Statistics of the Canonical Dataset

Computing exact numerical summaries for each sensor feature.


In [3]:
features = ['pH', 'TDS_ppm', 'Turbidity_NTU', 'Temperature_C']
desc = df[features].describe(percentiles=[0.25, 0.50, 0.75]).T
desc['median'] = df[features].median()

print("Overall Numerical Feature Statistics:")
display(desc[['count', 'mean', 'std', 'min', '25%', 'median', '75%', 'max']].round(4))

print("\nPer-Class Means:")
display(df.groupby('Label')[features].mean().round(4))


Overall Numerical Feature Statistics:


,count,mean,std,min,25%,median,75%,max
pH,100800.0,7.5939,0.7995,6.09,7.01,7.38,8.100,9.96
TDS_ppm,100800.0,692.6470,647.3319,28.60,163.00,472.85,1023.325,2806.10
Turbidity_NTU,100800.0,11.3067,9.8684,0.00,1.92,9.97,18.540,72.75
Temperature_C,100800.0,24.8835,2.6881,16.56,23.09,25.04,26.930,31.51



Per-Class Means:


,pH,TDS_ppm,Turbidity_NTU,Temperature_C
Label,,,,
Safe,7.1670,118.3442,1.2833,22.2075
Unsafe,7.7769,938.7768,15.6025,26.0304


## 3. Explanation of Inconsistencies Identified Across Past Reports

### Forensic Investigation of Previous Discrepancies:

| Field | Historical Discrepancy Found in Some Reports | Verified Ground Truth | Root Cause & Resolution |
| :--- | :--- | :--- | :--- |
| **File Size** | 4,374,321 bytes (4.17 MB) | **2,964,409 bytes (2.83 MB)** | When `src/add_engineered_columns.py` was temporarily tested, it appended 5 temporal columns, expanding the file to 4.17 MB. `src/revert_engineered_columns.py` cleanly restored the original 5-column CSV (2,964,409 bytes, SHA-256 `90f9c7cb...`). The 4.17 MB text was a stale residual mention. |
| **Class Block Ordering** | "Rows 0 to 70,559: Unsafe; 70,560 to 100,799: Safe" | **Rows 0 to 30,239: Safe; 30,240 to 100,799: Unsafe** | Human typographical inversion in report text: 70,560 was the *count* of unsafe rows, but `Safe` physically precedes `Unsafe` in the CSV. Exact transition occurs at index **30240**. |
| **TDS Mean** | 557.54 ppm (in text of `phase4_analysis_handoff.md`) | **692.65 ppm** | Typographical error in markdown narrative text. True dataset mean is 692.65 ppm (Safe mean: 118.34 ppm, Unsafe mean: 938.78 ppm). |


## 4. Audit of the Empirical Threshold Rule vs. Statutory Standards

### Distinction between Empirical Rule and Official Standards:

1. **Empirical Rule Inferred from Dataset**:
   $$\text{Safe} \iff (\text{TDS} \le 211.0\,\text{ppm}) \land (\text{Turbidity} \le 4.45\,\text{NTU}) \land (6.50 \le \text{pH} \le 7.80)$$
2. **Official Statutory Standards**:
   - **WHO Guidelines for Drinking-water Quality (4th ed.)**:
     - Turbidity: Ideally < 1 NTU for effective chlorination/disinfection; operational maximum: 5 NTU.
     - TDS: Palatable < 300–600 mg/L; no mandatory health-based limit, but taste cutoff is 500–1000 mg/L.
     - pH: 6.5 to 8.5.
   - **BIS (IS 10500: 2012)**:
     - Turbidity: Acceptable limit = 1 NTU; Permissible limit = 5 NTU.
     - TDS: Acceptable limit = 500 mg/L; Permissible limit = 2000 mg/L.
     - pH: 6.5 to 8.5.
   - **US EPA (Secondary Drinking Water Regulations)**:
     - Turbidity: 0.5 to 1.0 NTU.
     - TDS: Secondary Maximum Contaminant Level = 500 mg/L.
     - pH: 6.5 to 8.5.

> [!WARNING]
> The thresholds $TDS \le 211.0$ ppm and $\text{Turbidity} \le 4.45$ NTU are **strictly empirical artifacts** of this specific dataset. They must **never** be attributed as official statutory limits of WHO, BIS, or EPA.


In [4]:
safe_gt = (df['Label'] == 'Safe')
rule_pred_safe = (
    (df['TDS_ppm'] <= 211.0) & 
    (df['Turbidity_NTU'] <= 4.45) & 
    (df['pH'] >= 6.5) & 
    (df['pH'] <= 7.8)
)

exact_matches = int((safe_gt == rule_pred_safe).sum())
mismatches = int((safe_gt != rule_pred_safe).sum())

print(f"Total Observations : {len(df):,}")
print(f"Rule Matches       : {exact_matches:,} ({exact_matches/len(df)*100:.4f}%)")
print(f"Rule Mismatches    : {mismatches} ({mismatches/len(df)*100:.4f}%)")

print("\nThe 14 Boundary Edge Cases:")
edge_cases = df[safe_gt != rule_pred_safe]
display(edge_cases)


Total Observations : 100,800
Rule Matches       : 100,786 (99.9861%)
Rule Mismatches    : 14 (0.0139%)

The 14 Boundary Edge Cases:


,pH,TDS_ppm,Turbidity_NTU,Temperature_C,Label
50692,7.24,208.1,4.34,23.96,Unsafe
50943,7.29,202.2,4.43,25.88,Unsafe
51273,7.12,209.9,4.43,27.23,Unsafe
52023,7.51,207.4,4.28,24.62,Unsafe
52602,7.48,209.3,3.97,27.96,Unsafe
53867,7.54,207.9,4.03,26.51,Unsafe
54610,7.68,207.0,4.33,23.67,Unsafe
55037,7.05,192.3,3.95,24.26,Unsafe
55678,6.79,208.5,4.05,26.51,Unsafe
56427,7.01,209.8,3.75,24.50,Unsafe


## 5. Corrected Cross-Validation Methodology

### Methodological Correction:
- **Previous Procedure**: In the initial exploratory script, 5-fold cross-validation was run on the entire dataset ($N=100,800$), including samples from the held-out test partition in the CV fold pool.
- **Corrected Standard ML Procedure**:
  1. Partition dataset first into **80% Training ($N=80,640$)** and **20% Testing ($N=20,160$)** using stratified sampling (`random_state=42`).
  2. Reserve the 20% test partition as an untouched, completely independent evaluation set.
  3. Execute 5-fold Stratified Cross-Validation **exclusively on the 80% training partition**.
  4. Ensure preprocessing (`StandardScaler`) is fitted strictly on each training fold.


In [5]:
target_map = {'Safe': 0, 'Unsafe': 1}
X = df[features].copy()
y = df['Label'].map(target_map)

# 1. 80/20 Stratified Partition
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Partition (80%) : {len(X_train):,} samples (Safe={list(y_train).count(0):,}, Unsafe={list(y_train).count(1):,})")
print(f"Test Partition (20%)     : {len(X_test):,} samples (Safe={list(y_test).count(0):,}, Unsafe={list(y_test).count(1):,})")

# 2. 5-Fold Stratified CV EXCLUSIVELY on Training Partition
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Dummy (Most Frequent)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, criterion='gini', random_state=42, n_jobs=-1)
}

cv_recomputed = []

for name, model in models.items():
    accs, macro_f1s, rec_safes, rec_unsafes = [], [], [], []
    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        
        accs.append(accuracy_score(y_val, preds))
        macro_f1s.append(f1_score(y_val, preds, average='macro'))
        rec_safes.append(recall_score(y_val, preds, pos_label=0))
        rec_unsafes.append(recall_score(y_val, preds, pos_label=1))
        
    cv_recomputed.append({
        'Model': name,
        'CV Accuracy (Train Folds)': f"{np.mean(accs):.6f} ± {np.std(accs):.6f}",
        'CV Macro F1 (Train Folds)': f"{np.mean(macro_f1s):.6f} ± {np.std(macro_f1s):.6f}",
        'CV Safe Recall (Class 0)': f"{np.mean(rec_safes):.6f} ± {np.std(rec_safes):.6f}",
        'CV Unsafe Recall (Class 1)': f"{np.mean(rec_unsafes):.6f} ± {np.std(rec_unsafes):.6f}"
    })

cv_recomputed_df = pd.DataFrame(cv_recomputed)
print("\nRecomputed Cross-Validation Results (Strictly on 80,640 Training Samples):")
display(cv_recomputed_df)


Training Partition (80%) : 80,640 samples (Safe=24,192, Unsafe=56,448)
Test Partition (20%)     : 20,160 samples (Safe=6,048, Unsafe=14,112)



Recomputed Cross-Validation Results (Strictly on 80,640 Training Samples):


,Model,CV Accuracy (Train Folds),CV Macro F1 (Train Folds),CV Safe Recall (Class 0),CV Unsafe Recall (Class 1)
0,Dummy (Most Frequent),0.700000 ± 0.000030,0.411765 ± 0.000011,0.000000 ± 0.000000,1.000000 ± 0.000000
1,Logistic Regression,0.999678 ± 0.000132,0.999616 ± 0.000158,1.000000 ± 0.000000,0.999539 ± 0.000189
2,Decision Tree,0.999864 ± 0.000072,0.999838 ± 0.000086,0.999793 ± 0.000131,0.999894 ± 0.000066
3,Random Forest,0.999938 ± 0.000039,0.999926 ± 0.000047,0.999876 ± 0.000165,0.999965 ± 0.000043


In [6]:
# 3. Independent Evaluation on the Untouched Held-Out Test Partition (20,160 samples)
test_records = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    cm = confusion_matrix(list(y_test), [int(p) for p in y_pred])
    tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
    
    test_records.append({
        'Model': name,
        'Test Accuracy': float(accuracy_score(y_test, y_pred)),
        'Test Macro F1': float(f1_score(y_test, y_pred, average='macro')),
        'Test Safe Recall': float(recall_score(y_test, y_pred, pos_label=0)),
        'Test Unsafe Recall': float(recall_score(y_test, y_pred, pos_label=1)),
        'TN (Safe)': tn,
        'FP (Clean -> Unsafe)': fp,
        'FN (Unsafe -> Safe)': fn,
        'TP (Unsafe)': tp
    })

test_eval_df = pd.DataFrame(test_records)
print("\nIndependent Held-Out Test Evaluation (20,160 Unseen Samples):")
display(test_eval_df)



Independent Held-Out Test Evaluation (20,160 Unseen Samples):


,Model,Test Accuracy,Test Macro F1,Test Safe Recall,Test Unsafe Recall,TN (Safe),FP (Clean -> Unsafe),FN (Unsafe -> Safe),TP (Unsafe)
0,Dummy (Most Frequent),0.700000,0.411765,0.000000,1.000000,0,6048,0,14112
1,Logistic Regression,0.999901,0.999882,1.000000,0.999858,6048,0,2,14110
2,Decision Tree,0.999901,0.999882,0.999835,0.999929,6047,1,1,14111
3,Random Forest,0.999901,0.999882,0.999835,0.999929,6047,1,1,14111


## 6. Audit Summary & Readiness for Phase 5

### Key Takeaways from the Phase 4.1 Audit:
1. **Canonical Dataset Established**:
   - File: `data/water_quality_dataset1.csv`
   - Exact size: **2,964,409 bytes**
   - SHA-256: `90f9c7cb7404fb69b655ee27293c91b82ee3a346712e06035f8259393835a04a`
   - Block Layout: `Safe` comes first (rows 0 to 30,239); `Unsafe` comes second (rows 30,240 to 100,799). Transition is at row **30240**.
2. **Empirical Rule Clarified**:
   - The boundary thresholds ($TDS \le 211.0$, $\text{Turbidity} \le 4.45$, $6.5 \le pH \le 7.8$) are purely empirical features of this dataset (matching 99.9861%), not statutory WHO/BIS numbers.
3. **Cross-Validation Corrected**:
   - 5-fold CV is strictly computed on the training partition ($N=80,640$) with no test contamination.
   - All standard deviations remain $\le 0.00015$, confirming stability.
4. **Final Model Confirmed**:
   - **Decision Tree (`max_depth=5`)** remains the confirmed production candidate for Phase 5 explainability and Phase 6 cloud deployment.
